# Complete Preprocessing Pipeline for snRNA-seq Data

**Module 00: Data Preprocessing**

---

## Overview

This notebook performs comprehensive preprocessing of single-nucleus RNA-seq data:

1. **Quality Control** - Cell and gene filtering with snRNA-seq-appropriate thresholds
2. **Normalization** - CPM normalization to 10,000 counts per cell
3. **Feature Selection** - Identify highly variable genes
4. **Scaling** - Stored in layer (not active)
5. **Dimensionality Reduction** - PCA on HVGs
6. **Batch Correction** - Harmony integration
7. **Clustering** - Leiden algorithm
8. **Cell Type Validation** - Canonical marker validation

**Data Layers:**
- `adata.X` - Log-normalized (active layer for analysis)
- `adata.layers['counts']` - Raw counts (for DESeq2)
- `adata.layers['normalized']` - Normalized counts
- `adata.layers['log1p']` - Log-normalized
- `adata.layers['scaled']` - Scaled data (optional)
- `adata.raw` - Full log-normalized matrix

**Configuration:** Dataset-specific settings loaded from `config/datasets.yaml`

---

**Author:** Gerald Gaitos  
**Date:** December 2025  
**Pipeline:** Cellular Senescence in Brain Aging

---

## Setup & Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check versions
print(f"scanpy version: {sc.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET SELECTION
# ═══════════════════════════════════════════════════════════════════════════════

DATASET = 'psychad_aging'  # ← CHANGE THIS FOR EACH COHORT

# Options:
#   - 'psychad_aging'
#   - 'psychad_ad'
#   - 'psychencode'
#   - 'mathys'
#   - 'australian'

print(f"Selected dataset: {DATASET}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# Path to config file (relative to notebook)
CONFIG_FILE = Path('../config/datasets.yaml')

# Load configuration
try:
    with open(CONFIG_FILE, 'r') as f:
        config_data = yaml.safe_load(f)
    
    if DATASET not in config_data['datasets']:
        raise ValueError(f"Dataset '{DATASET}' not found in config file")
    
    dataset_config = config_data['datasets'][DATASET]
    params = config_data['parameters']
    
    print(f"✓ Configuration loaded from: {CONFIG_FILE.name}")
    
except FileNotFoundError:
    raise FileNotFoundError(
        f"Config file not found: {CONFIG_FILE}\n"
        "Please ensure config/datasets.yaml exists and contains dataset configurations."
    )

# Display config
print(f"\nDataset: {DATASET}")
print(f"Description: {dataset_config.get('description', 'N/A')}")
print(f"Region: {dataset_config.get('region', 'N/A')}")
print(f"Samples: {dataset_config.get('n_samples', 'N/A')}")
print(f"Analysis type: {dataset_config.get('analysis_type', 'N/A')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTRACT CONFIGURATION PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════

# ───────────────────────────────────────────────────────────────────────────────
# Dataset-Specific
# ───────────────────────────────────────────────────────────────────────────────
INPUT_FILE = Path(dataset_config['input_file'])
BATCH_KEY = dataset_config['batch_key']
CELL_TYPE_COLUMN = dataset_config.get('cell_type_column')
SAMPLE_COLUMN = dataset_config.get('sample_column')
DONOR_COLUMN = dataset_config.get('donor_column')

# ───────────────────────────────────────────────────────────────────────────────
# QC Parameters
# ───────────────────────────────────────────────────────────────────────────────
MIN_GENES = params['qc']['min_genes']
MAX_GENES = params['qc']['max_genes']
MAX_MT_PERCENT = params['qc']['max_mt_percent']
MIN_CELLS = params['qc']['min_cells']

# ───────────────────────────────────────────────────────────────────────────────
# Normalization
# ───────────────────────────────────────────────────────────────────────────────
TARGET_SUM = params['normalization']['target_sum']

# ───────────────────────────────────────────────────────────────────────────────
# Feature Selection
# ───────────────────────────────────────────────────────────────────────────────
N_TOP_GENES = params['feature_selection']['n_top_genes']
HVG_FLAVOR = params['feature_selection']['flavor']
SUBSET_HVG = params['feature_selection']['subset']

# ───────────────────────────────────────────────────────────────────────────────
# Scaling
# ───────────────────────────────────────────────────────────────────────────────
PERFORM_SCALING = params['scaling']['perform_scaling']
SCALE_MAX_VALUE = params['scaling']['max_value']

# ───────────────────────────────────────────────────────────────────────────────
# Dimensionality Reduction
# ───────────────────────────────────────────────────────────────────────────────
N_PCS = params['pca']['n_pcs']
SVD_SOLVER = params['pca']['svd_solver']

# ───────────────────────────────────────────────────────────────────────────────
# Clustering
# ───────────────────────────────────────────────────────────────────────────────
N_NEIGHBORS = params['clustering']['n_neighbors']
LEIDEN_RESOLUTION = params['clustering']['leiden_resolution']
LEIDEN_FLAVOR = params['clustering']['leiden_flavor']
LEIDEN_N_ITERATIONS = params['clustering']['leiden_n_iterations']

# ───────────────────────────────────────────────────────────────────────────────
# Paths
# ───────────────────────────────────────────────────────────────────────────────
BASE_DIR = Path(params['paths']['base_dir'])
OUTPUT_DIR = BASE_DIR / params['paths']['output_subdir']
FIGURES_DIR = BASE_DIR / params['paths']['figures_subdir'] / params['paths']['module_figures']['preprocessing'] / DATASET

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Output file
OUTPUT_FILE = OUTPUT_DIR / f'{DATASET}_preprocessed.h5ad'

# ───────────────────────────────────────────────────────────────────────────────
# Random Seed
# ───────────────────────────────────────────────────────────────────────────────
RANDOM_SEED = params['random_seed']
np.random.seed(RANDOM_SEED)

# ───────────────────────────────────────────────────────────────────────────────
# Display Configuration
# ───────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("CONFIGURATION SUMMARY")
print("="*80)

print(f"\nInput:")
print(f"  File: {INPUT_FILE}")
print(f"  Batch key: {BATCH_KEY}")
print(f"  Cell type column: {CELL_TYPE_COLUMN}")

print(f"\nQC Thresholds (snRNA-seq):")
print(f"  {MIN_GENES} ≤ genes/cell ≤ {MAX_GENES}")
print(f"  MT% < {MAX_MT_PERCENT}%")
print(f"  Gene min cells: {MIN_CELLS}")

print(f"\nProcessing:")
print(f"  Normalization: {TARGET_SUM:.0e} counts/cell")
print(f"  HVGs: {N_TOP_GENES} (flavor: {HVG_FLAVOR})")
print(f"  PCA: {N_PCS} components")
print(f"  Scaling: {'Yes (stored in layer)' if PERFORM_SCALING else 'No'}")

print(f"\nClustering:")
print(f"  k-neighbors: {N_NEIGHBORS}")
print(f"  Leiden resolution: {LEIDEN_RESOLUTION}")

print(f"\nOutput:")
print(f"  Processed data: {OUTPUT_FILE}")
print(f"  Figures: {FIGURES_DIR}")

print(f"\nRandom seed: {RANDOM_SEED}")
print("="*80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE SETTINGS
# ═══════════════════════════════════════════════════════════════════════════════

FIGURE_DPI = params['visualization']['figure_dpi']

# Scanpy settings
sc.settings.set_figure_params(
    dpi=FIGURE_DPI,
    dpi_save=FIGURE_DPI,
    frameon=False,
    figsize=(6, 6)
)
sc.settings.figdir = FIGURES_DIR

# Publication-quality plotting style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.dpi': FIGURE_DPI,
    'savefig.dpi': FIGURE_DPI,
    'savefig.bbox': 'tight',
    'savefig.transparent': False,
})

print(f"✓ Figure settings configured (DPI: {FIGURE_DPI})")
print(f"  Figures will be saved to: {FIGURES_DIR}")

---

## Step 1: Load Data

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════

print("="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# Verify file exists
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

# Load data
print(f"\nLoading: {INPUT_FILE.name}...")
adata = sc.read_h5ad(INPUT_FILE)

print(f"\n✓ Data loaded successfully")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Matrix type: {type(adata.X)}")
print(f"  Memory: {adata.X.data.nbytes / 1e9:.2f} GB" if hasattr(adata.X, 'data') else '')

# Display metadata
print(f"\nMetadata columns ({len(adata.obs.columns)}):")
for col in sorted(adata.obs.columns):
    print(f"  - {col}")

# Verify required columns exist
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found in adata.obs")

print(f"\n✓ Batch variable '{BATCH_KEY}' found")
print(f"  Number of batches: {adata.obs[BATCH_KEY].nunique()}")

if CELL_TYPE_COLUMN and CELL_TYPE_COLUMN in adata.obs.columns:
    print(f"\n✓ Cell type column '{CELL_TYPE_COLUMN}' found")
    print(f"  Number of cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")
else:
    print(f"\n⚠ Cell type column '{CELL_TYPE_COLUMN}' not found")
    print("  Cell type validation will be skipped")

---

## Step 2: Calculate QC Metrics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALCULATE QC METRICS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 2: CALCULATING QC METRICS")
print("="*80)

# Identify mitochondrial genes
print("\nIdentifying mitochondrial genes...")
adata.var['mt'] = adata.var_names.str.startswith('MT-')
n_mt_genes = adata.var['mt'].sum()
print(f"✓ Found {n_mt_genes} mitochondrial genes")

# Calculate QC metrics
print("\nCalculating quality control metrics...")
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("✓ QC metrics calculated:")
print("  - n_genes_by_counts: genes detected per cell")
print("  - total_counts: total UMI counts per cell")
print("  - pct_counts_mt: % mitochondrial expression")

# Summary statistics
print("\nQC Summary (before filtering):")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"\n  Genes/cell:")
print(f"    Median: {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"    Mean: {adata.obs['n_genes_by_counts'].mean():.0f}")
print(f"    Range: [{adata.obs['n_genes_by_counts'].min():.0f}, {adata.obs['n_genes_by_counts'].max():.0f}]")
print(f"\n  Counts/cell:")
print(f"    Median: {adata.obs['total_counts'].median():.0f}")
print(f"    Mean: {adata.obs['total_counts'].mean():.0f}")
print(f"\n  MT%:")
print(f"    Median: {adata.obs['pct_counts_mt'].median():.2f}%")
print(f"    Mean: {adata.obs['pct_counts_mt'].mean():.2f}%")
print(f"    Range: [{adata.obs['pct_counts_mt'].min():.2f}%, {adata.obs['pct_counts_mt'].max():.2f}%]")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE QC METRICS (BEFORE FILTERING)
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCreating QC visualizations...")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f'{DATASET} - QC Metrics Before Filtering', 
             fontsize=14, fontweight='bold', y=1.02)

# Genes per cell
axes[0].hist(adata.obs['n_genes_by_counts'], bins=100, color='#4A90E2', alpha=0.7, edgecolor='black')
axes[0].axvline(MIN_GENES, color='red', linestyle='--', linewidth=2, label=f'Min: {MIN_GENES}')
axes[0].axvline(MAX_GENES, color='red', linestyle='--', linewidth=2, label=f'Max: {MAX_GENES}')
axes[0].set_xlabel('Genes detected per cell', fontweight='bold')
axes[0].set_ylabel('Number of cells', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Total counts (log scale)
axes[1].hist(np.log10(adata.obs['total_counts']), bins=100, color='#50C878', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('log₁₀(Total counts per cell)', fontweight='bold')
axes[1].set_ylabel('Number of cells', fontweight='bold')
axes[1].grid(alpha=0.3)

# Mitochondrial %
axes[2].hist(adata.obs['pct_counts_mt'], bins=100, color='#E24A4A', alpha=0.7, edgecolor='black')
axes[2].axvline(MAX_MT_PERCENT, color='red', linestyle='--', linewidth=2, 
                label=f'Max: {MAX_MT_PERCENT}%')
axes[2].set_xlabel('% Mitochondrial', fontweight='bold')
axes[2].set_ylabel('Number of cells', fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'{DATASET}_qc_before_filtering.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / f'{DATASET}_qc_before_filtering.png', bbox_inches='tight')
plt.show()

print(f"✓ QC plots saved to: {FIGURES_DIR}")

---

## Step 3: Quality Control Filtering

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# QUALITY CONTROL FILTERING
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 3: QUALITY CONTROL FILTERING")
print("="*80)

# Record initial counts
n_cells_before = adata.n_obs
n_genes_before = adata.n_vars

print(f"\nBefore filtering:")
print(f"  Cells: {n_cells_before:,}")
print(f"  Genes: {n_genes_before:,}")

# ───────────────────────────────────────────────────────────────────────────────
# Filter Genes
# ───────────────────────────────────────────────────────────────────────────────
print(f"\nFiltering genes (must be expressed in ≥{MIN_CELLS} cells)...")
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"✓ Genes remaining: {adata.n_vars:,}")

# ───────────────────────────────────────────────────────────────────────────────
# Filter Cells - Minimum Genes
# ───────────────────────────────────────────────────────────────────────────────
print(f"\nFiltering cells (minimum genes: ≥{MIN_GENES})...")
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
print(f"✓ Cells remaining: {adata.n_obs:,}")

# ───────────────────────────────────────────────────────────────────────────────
# Filter Cells - Maximum Genes (Doublets)
# ───────────────────────────────────────────────────────────────────────────────
print(f"\nFiltering cells (maximum genes: ≤{MAX_GENES})...")
adata = adata[adata.obs['n_genes_by_counts'] <= MAX_GENES, :].copy()
print(f"✓ Cells remaining: {adata.n_obs:,}")

# ───────────────────────────────────────────────────────────────────────────────
# Filter Cells - Mitochondrial % (Strict for snRNA-seq)
# ───────────────────────────────────────────────────────────────────────────────
print(f"\nFiltering cells (mitochondrial %: <{MAX_MT_PERCENT}%)...")
print(f"Note: Strict threshold appropriate for nuclear RNA-seq")
adata = adata[adata.obs['pct_counts_mt'] < MAX_MT_PERCENT, :].copy()
print(f"✓ Cells remaining: {adata.n_obs:,}")

# ───────────────────────────────────────────────────────────────────────────────
# Filtering Summary
# ───────────────────────────────────────────────────────────────────────────────
n_cells_removed = n_cells_before - adata.n_obs
n_genes_removed = n_genes_before - adata.n_vars
pct_cells_removed = (n_cells_removed / n_cells_before) * 100
pct_genes_removed = (n_genes_removed / n_genes_before) * 100

print("\n" + "-"*80)
print("FILTERING SUMMARY")
print("-"*80)
print(f"\nCells:")
print(f"  Before: {n_cells_before:,}")
print(f"  After:  {adata.n_obs:,}")
print(f"  Removed: {n_cells_removed:,} ({pct_cells_removed:.1f}%)")

print(f"\nGenes:")
print(f"  Before: {n_genes_before:,}")
print(f"  After:  {adata.n_vars:,}")
print(f"  Removed: {n_genes_removed:,} ({pct_genes_removed:.1f}%)")

print(f"\nFinal dataset:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Matrix size: {adata.n_obs:,} × {adata.n_vars:,}")

---

## Step 4: Preserve Raw Counts

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE RAW COUNTS TO LAYER
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 4: PRESERVING RAW COUNTS")
print("="*80)

print("\nSaving raw counts to layer...")
adata.layers['counts'] = adata.X.copy()

print("✓ Raw counts saved to: adata.layers['counts']")
print("  Purpose: Essential for DESeq2 differential expression")
print("  Note: DESeq2 requires integer counts (not normalized)")

# Verify counts
if hasattr(adata.layers['counts'], 'data'):
    sample_values = adata.layers['counts'].data[:100]
else:
    sample_values = adata.layers['counts'][:5, :5]

print(f"\n  Sample values: {sample_values[:5]}")
print(f"  Min: {adata.layers['counts'].min()}")
print(f"  Max: {adata.layers['counts'].max()}")

---

## Step 5: Normalization

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NORMALIZATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 5: NORMALIZATION")
print("="*80)

print(f"\nNormalizing to {TARGET_SUM:.0e} counts per cell...")
print("Method: CPM (Counts Per Million) normalization")

# Normalize
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)

# Save normalized counts
adata.layers['normalized'] = adata.X.copy()

# Verify normalization
mean_counts = adata.X.sum(axis=1).mean()

print(f"\n✓ Normalization complete")
print(f"  Target: {TARGET_SUM:.0e} counts/cell")
print(f"  Actual mean: {mean_counts:.2f} counts/cell")
print(f"  Saved to: adata.layers['normalized']")

# Check distribution
if hasattr(adata.X, 'toarray'):
    sample_sums = adata.X.sum(axis=1).A1[:1000]
else:
    sample_sums = adata.X.sum(axis=1)[:1000]

print(f"\n  Sample cell totals (first 1000):")
print(f"    Min: {sample_sums.min():.0f}")
print(f"    Max: {sample_sums.max():.0f}")
print(f"    Median: {np.median(sample_sums):.0f}")

---

## Step 6: Log Transformation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LOG TRANSFORMATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 6: LOG TRANSFORMATION")
print("="*80)

print("\nApplying log1p transformation...")
print("Formula: log(x + 1)")

# Log transform
sc.pp.log1p(adata)

# Save log-normalized counts
adata.layers['log1p'] = adata.X.copy()

print("\n✓ Log transformation complete")
print("  Saved to: adata.layers['log1p']")
print("  Current adata.X: log-normalized (active layer)")

# Check values
if hasattr(adata.X, 'data'):
    print(f"\n  Value range:")
    print(f"    Min: {adata.X.data.min():.3f}")
    print(f"    Max: {adata.X.data.max():.3f}")
else:
    print(f"\n  Value range:")
    print(f"    Min: {adata.X.min():.3f}")
    print(f"    Max: {adata.X.max():.3f}")

---

## Step 7: Highly Variable Genes

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HIGHLY VARIABLE GENE SELECTION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 7: HIGHLY VARIABLE GENE SELECTION")
print("="*80)

print(f"\nIdentifying top {N_TOP_GENES} highly variable genes...")
print(f"Method: {HVG_FLAVOR}")

# Find HVGs
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_TOP_GENES,
    flavor=HVG_FLAVOR,
    subset=SUBSET_HVG
)

n_hvgs = adata.var['highly_variable'].sum()

print(f"\n✓ HVG selection complete")
print(f"  HVGs identified: {n_hvgs:,}")
print(f"  Total genes kept: {adata.n_vars:,}")
print(f"  Subset to HVGs: {SUBSET_HVG}")

if not SUBSET_HVG:
    print("\n  Note: All genes retained in dataset")
    print("  HVGs will be used for PCA only")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE HVGs
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCreating HVG plot...")

sc.pl.highly_variable_genes(adata, save=f'_{DATASET}_hvgs.pdf')

print(f"✓ HVG plot saved")

---

## Step 8: Save Raw Matrix

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE RAW (FULL LOG-NORMALIZED MATRIX)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 8: PRESERVING FULL MATRIX")
print("="*80)

print("\nSaving full log-normalized matrix...")
adata.raw = adata

print("✓ Full matrix saved to: adata.raw")
print("  Purpose: Used for visualization and marker gene expression")
print("  Contains: All genes (not just HVGs)")
print(f"  Shape: {adata.raw.shape[0]:,} cells × {adata.raw.shape[1]:,} genes")

---

## Step 9: Scaling

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SCALING (SAVE TO LAYER, DON'T OVERWRITE ACTIVE)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 9: SCALING DATA")
print("="*80)

if PERFORM_SCALING:
    print("\nScaling data...")
    print("Method: Z-score (mean=0, variance=1)")
    print(f"Max value: {SCALE_MAX_VALUE} (clip outliers)")
    
    # Scale data (temporarily modifies adata.X)
    sc.pp.scale(adata, max_value=SCALE_MAX_VALUE)
    
    # Save scaled to layer
    adata.layers['scaled'] = adata.X.copy()
    
    # RESTORE log-normalized to active layer
    adata.X = adata.layers['log1p'].copy()
    
    print("\n✓ Scaling complete")
    print("  Scaled data saved to: adata.layers['scaled']")
    print("  Active layer (adata.X): log-normalized (RESTORED)")
    print("\n  Why restore log-normalized?")
    print("    - Harmony works better on log-normalized data")
    print("    - UMAP/clustering more robust with log-normalized")
    print("    - Scaled data available if specific tools need it")
else:
    print("\n⊗ Scaling skipped (PERFORM_SCALING = False)")
    print("  Active layer remains: log-normalized")

---

## Current Data Structure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DISPLAY CURRENT DATA STRUCTURE
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("CURRENT DATA STRUCTURE")
print("="*80)

print("\nData Layers:")
print(f"  adata.X (active):              log-normalized")
print(f"  adata.layers['counts']:        raw counts (for DESeq2)")
print(f"  adata.layers['normalized']:    normalized counts")
print(f"  adata.layers['log1p']:         log-normalized")
if PERFORM_SCALING:
    print(f"  adata.layers['scaled']:        scaled data")
print(f"  adata.raw:                     full log-normalized matrix")

print(f"\nDimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  HVGs: {adata.var['highly_variable'].sum():,}")

print(f"\nMemory:")
if hasattr(adata.X, 'data'):
    mem_x = adata.X.data.nbytes / 1e9
    print(f"  adata.X: {mem_x:.2f} GB")

total_layers = 0
for layer_name in adata.layers.keys():
    if hasattr(adata.layers[layer_name], 'data'):
        total_layers += adata.layers[layer_name].data.nbytes / 1e9
if total_layers > 0:
    print(f"  Layers: {total_layers:.2f} GB")

---

## Step 10: PCA

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PRINCIPAL COMPONENT ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 10: PRINCIPAL COMPONENT ANALYSIS")
print("="*80)

print(f"\nComputing PCA...")
print(f"  Components: {N_PCS}")
print(f"  Solver: {SVD_SOLVER}")
print(f"  Use HVGs: {adata.var['highly_variable'].sum():,} genes")

# Compute PCA on log-normalized data
sc.tl.pca(
    adata,
    n_comps=N_PCS,
    svd_solver=SVD_SOLVER,
    use_highly_variable=True,
    random_state=RANDOM_SEED
)

variance_explained = adata.uns['pca']['variance_ratio'].sum()

print(f"\n✓ PCA complete")
print(f"  PCA stored in: adata.obsm['X_pca']")
print(f"  Variance explained: {variance_explained:.1%}")
print(f"  PC1 variance: {adata.uns['pca']['variance_ratio'][0]:.2%}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE PCA
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCreating PCA variance plot...")

sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True, save=f'_{DATASET}_pca_variance.pdf')

print(f"✓ PCA variance plot saved")

---

## Step 11: Batch Correction (Harmony)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# BATCH CORRECTION WITH HARMONY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 11: BATCH CORRECTION WITH HARMONY")
print("="*80)

print(f"\nRunning Harmony integration...")
print(f"  Batch key: {BATCH_KEY}")
print(f"  Number of batches: {adata.obs[BATCH_KEY].nunique()}")
print(f"  PCA components: {N_PCS}")

# Display batch distribution
batch_counts = adata.obs[BATCH_KEY].value_counts()
print(f"\n  Batch sizes:")
print(f"    Min: {batch_counts.min():,} cells")
print(f"    Max: {batch_counts.max():,} cells")
print(f"    Median: {batch_counts.median():.0f} cells")

# Run Harmony
print("\n  Integrating batches...")
sce.pp.harmony_integrate(
    adata,
    key=BATCH_KEY,
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=RANDOM_SEED
)

print("\n✓ Harmony integration complete")
print("  Corrected PCA stored in: adata.obsm['X_pca_harmony']")
print("  Original PCA preserved in: adata.obsm['X_pca']")

---

## Step 12: Neighbors & UMAP

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# NEIGHBOR GRAPH & UMAP
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 12: NEIGHBOR GRAPH & UMAP")
print("="*80)

# ───────────────────────────────────────────────────────────────────────────────
# Build Neighbor Graph
# ───────────────────────────────────────────────────────────────────────────────
print(f"\nBuilding neighbor graph...")
print(f"  k-neighbors: {N_NEIGHBORS}")
print(f"  PCA components: {N_PCS}")
print(f"  Using: Harmony-corrected PCA")

sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS,
    use_rep='X_pca_harmony',
    random_state=RANDOM_SEED
)

print("\n✓ Neighbor graph computed")

# ───────────────────────────────────────────────────────────────────────────────
# Compute UMAP
# ───────────────────────────────────────────────────────────────────────────────
print("\nComputing UMAP embedding...")

sc.tl.umap(adata, random_state=RANDOM_SEED)

print("\n✓ UMAP complete")
print("  UMAP coordinates stored in: adata.obsm['X_umap']")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE BATCH CORRECTION
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCreating batch correction visualization...")

sc.pl.umap(
    adata,
    color=BATCH_KEY,
    title='Batch Effects After Harmony',
    save=f'_{DATASET}_batch_correction.pdf'
)

print(f"✓ Batch correction plot saved")

---

## Step 13: Leiden Clustering

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LEIDEN CLUSTERING
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 13: LEIDEN CLUSTERING")
print("="*80)

print(f"\nRunning Leiden clustering...")
print(f"  Resolution: {LEIDEN_RESOLUTION}")
print(f"  Flavor: {LEIDEN_FLAVOR}")
print(f"  Iterations: {LEIDEN_N_ITERATIONS}")

# Run Leiden
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    flavor=LEIDEN_FLAVOR,
    n_iterations=LEIDEN_N_ITERATIONS,
    random_state=RANDOM_SEED
)

n_clusters = adata.obs['leiden'].nunique()

print(f"\n✓ Clustering complete")
print(f"  Clusters identified: {n_clusters}")
print(f"  Cluster sizes:")

cluster_counts = adata.obs['leiden'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    pct = (count / len(adata)) * 100
    print(f"    Cluster {cluster}: {count:,} cells ({pct:.1f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VISUALIZE CLUSTERS
# ═══════════════════════════════════════════════════════════════════════════════

print("\nCreating cluster visualization...")

sc.pl.umap(
    adata,
    color='leiden',
    legend_loc='on data',
    title=f'Leiden Clusters (n={n_clusters})',
    save=f'_{DATASET}_leiden_clusters.pdf'
)

print(f"✓ Cluster plot saved")

---

## Step 14: Cell Type Validation

**Yang-style validation:** Clean, publication-ready dotplot with canonical markers

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL TYPE VALIDATION (YANG STYLE)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 14: CELL TYPE VALIDATION")
print("="*80)

if CELL_TYPE_COLUMN is None or CELL_TYPE_COLUMN not in adata.obs.columns:
    print("\n⚠ Cell type column not found. Skipping validation.")
    print(f"  Expected column: {CELL_TYPE_COLUMN}")
    print("  Available columns:", list(adata.obs.columns))
    SKIP_VALIDATION = True
else:
    SKIP_VALIDATION = False
    
    print(f"\nCell type column: {CELL_TYPE_COLUMN}")
    
    # ───────────────────────────────────────────────────────────────────────────
    # Display cell type distribution
    # ───────────────────────────────────────────────────────────────────────────
    n_types = adata.obs[CELL_TYPE_COLUMN].nunique()
    print(f"\nCell types found: {n_types}")
    
    cell_type_counts = adata.obs[CELL_TYPE_COLUMN].value_counts()
    print("\nCell type distribution:")
    for ct, count in cell_type_counts.items():
        pct = (count / len(adata)) * 100
        print(f"  {ct}: {count:,} cells ({pct:.1f}%)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CANONICAL MARKERS (2-3 PER TYPE)
# ═══════════════════════════════════════════════════════════════════════════════

if not SKIP_VALIDATION:
    # Load markers from config
    CANONICAL_MARKERS = config_data['canonical_markers']
    
    print(f"\nCanonical markers loaded from config:")
    for ct, markers in CANONICAL_MARKERS.items():
        print(f"  {ct}: {markers}")
    
    # ───────────────────────────────────────────────────────────────────────────
    # Filter to available markers
    # ───────────────────────────────────────────────────────────────────────────
    print(f"\nFiltering to markers present in dataset...")
    
    markers_filtered = {
        ct: [g for g in genes if g in adata.var_names]
        for ct, genes in CANONICAL_MARKERS.items()
    }
    markers_filtered = {k: v for k, v in markers_filtered.items() if len(v) > 0}
    
    print(f"\nMarkers available:")
    total_markers = 0
    for ct, genes in markers_filtered.items():
        print(f"  {ct}: {', '.join(genes)}")
        total_markers += len(genes)
    
    print(f"\nTotal markers for validation: {total_markers}")
    
    if total_markers == 0:
        print("\n⚠ No canonical markers found in dataset")
        SKIP_VALIDATION = True

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL TYPE ORDER
# ═══════════════════════════════════════════════════════════════════════════════

if not SKIP_VALIDATION:
    # Define desired order (customize as needed)
    celltype_order = [
        'Excitatory', 'Inhibitory',
        'Astrocyte', 'Oligodendrocyte', 'OPC',
        'Microglia',
        'Endothelial', 'Pericyte', 'VLMC', 'VSMC',
    ]
    
    # Filter to cell types present in data
    available_celltypes = adata.obs[CELL_TYPE_COLUMN].unique()
    celltype_order = [ct for ct in celltype_order if ct in available_celltypes]
    
    # Add any cell types not in our predefined order
    extra_celltypes = [ct for ct in available_celltypes if ct not in celltype_order]
    celltype_order.extend(sorted(extra_celltypes))
    
    print(f"\nCell type order for plotting: {celltype_order}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MARKER VALIDATION DOTPLOT (YANG STYLE)
# ═══════════════════════════════════════════════════════════════════════════════

if not SKIP_VALIDATION:
    print("\nCreating marker validation dotplot...")
    print("Style: Clean, publication-ready (Yang et al.)")
    
    # Create figure
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Create dotplot
    sc.pl.dotplot(
        adata,
        var_names=markers_filtered,
        groupby=CELL_TYPE_COLUMN,
        categories_order=celltype_order,
        standard_scale='var',
        cmap='Reds',
        dot_min=0.1,
        dot_max=0.9,
        ax=ax,
        show=False,
        title=''
    )
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{DATASET}_marker_validation_dotplot.pdf', 
                bbox_inches='tight', dpi=300)
    plt.savefig(FIGURES_DIR / f'{DATASET}_marker_validation_dotplot.png', 
                bbox_inches='tight', dpi=300)
    plt.show()
    
    print(f"\n✓ Marker validation dotplot saved")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# UMAP WITH CELL TYPES
# ═══════════════════════════════════════════════════════════════════════════════

if not SKIP_VALIDATION:
    print("\nCreating UMAP with cell types...")
    
    sc.pl.umap(
        adata,
        color=CELL_TYPE_COLUMN,
        legend_loc='right margin',
        frameon=False,
        save=f'_{DATASET}_celltypes.pdf'
    )
    
    print(f"✓ Cell type UMAP saved")
    
    print("\n" + "="*80)
    print("✓ CELL TYPE VALIDATION COMPLETE")
    print("="*80)

---

## Step 15: Save Preprocessed Data

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SAVE PREPROCESSED DATA
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("STEP 15: SAVING PREPROCESSED DATA")
print("="*80)

print(f"\nSaving to: {OUTPUT_FILE}")

# Save
adata.write(OUTPUT_FILE)

# Get file size
file_size_gb = OUTPUT_FILE.stat().st_size / 1e9

print(f"\n✓ File saved successfully")
print(f"  Location: {OUTPUT_FILE}")
print(f"  Size: {file_size_gb:.2f} GB")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

---

## Preprocessing Summary

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("✓ PREPROCESSING PIPELINE COMPLETE")
print("="*80)

print(f"\nDataset: {DATASET}")
print(f"Description: {dataset_config.get('description', 'N/A')}")
print(f"Region: {dataset_config.get('region', 'N/A')}")

print(f"\nFinal Dimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  HVGs: {adata.var['highly_variable'].sum():,}")
print(f"  Leiden clusters: {adata.obs['leiden'].nunique()}")
if not SKIP_VALIDATION:
    print(f"  Cell types: {adata.obs[CELL_TYPE_COLUMN].nunique()}")

print(f"\nData Layers:")
print(f"  adata.X:                       log-normalized (active)")
print(f"  adata.layers['counts']:        raw counts (for DESeq2)")
print(f"  adata.layers['normalized']:    normalized counts")
print(f"  adata.layers['log1p']:         log-normalized")
if PERFORM_SCALING:
    print(f"  adata.layers['scaled']:        scaled data")
print(f"  adata.raw:                     full log-normalized matrix")

print(f"\nEmbeddings:")
print(f"  adata.obsm['X_pca']:           PCA ({N_PCS} components)")
print(f"  adata.obsm['X_pca_harmony']:   Harmony-corrected PCA")
print(f"  adata.obsm['X_umap']:          UMAP coordinates")

print(f"\nClustering:")
print(f"  adata.obs['leiden']:           {adata.obs['leiden'].nunique()} clusters")

print(f"\nOutput Files:")
print(f"  Processed data: {OUTPUT_FILE.name}")
print(f"  Figures: {FIGURES_DIR.name}/")

print(f"\nReady for:")
print(f"  ✓ Module 01: Senescence scoring (senepy)")
print(f"  ✓ Module 02: Glial subclustering")
print(f"  ✓ Module 03-07: Downstream analyses")

print("\n" + "="*80)
print(f"Preprocessing completed: {DATASET}")
print("="*80)